In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing import image
from PIL import ImageFile

print("All modules imported successfully.")

All modules imported successfully.


In [3]:
# ==============================
# Step 1: Load dataset
# ==============================
data_dir = "../../data sheets/training_set"

# Allow truncated images (prevents load errors)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Data generator with VGG16 preprocessing
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

batch_size = 32
img_size = (224, 224)

# Load images and labels from directory
generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",  # labels as integers
    shuffle=True
)

Found 5000 images belonging to 5 classes.


In [4]:
# ==============================
# Step 2: Feature extraction using VGG16
# ==============================
base_model = VGG16(weights="imagenet", include_top=False, pooling="avg")

features = []
labels = []

for i in range(len(generator)):
    try:
        x_batch, y_batch = generator[i]  # load batch
        feat_batch = base_model.predict(x_batch, verbose=0)  # extract features
        features.append(feat_batch)
        labels.append(y_batch)

        # Stop when all images processed
        if (i + 1) * batch_size >= generator.n:
            break

    except Exception as e:
        print(f" Skipping batch {i} due to error: {e}")
        continue

# Combine all features and labels
X = np.vstack(features)
y = np.hstack(labels)

print("Feature shape:", X.shape)
print("Labels shape:", y.shape)

Feature shape: (5000, 512)
Labels shape: (5000,)


In [5]:
# ==============================
# Step 3: Train SVM classifier
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create SVM model
clf = SVC(kernel="rbf", C=10, gamma="scale", random_state=42)

# Train
print(" Training SVM classifier...")
clf.fit(X_train, y_train)
print(" Training complete.")

 Training SVM classifier...
 Training complete.


In [6]:
# ==============================
# Step 4: Evaluate model
# ==============================
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\n Accuracy:", round(accuracy * 100, 2), "%")

# Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



 Accuracy: 44.2 %

Classification Report:
              precision    recall  f1-score   support

         0.0       0.39      0.43      0.41       200
         1.0       0.49      0.53      0.51       200
         2.0       0.39      0.33      0.36       200
         3.0       0.52      0.51      0.51       200
         4.0       0.41      0.41      0.41       200

    accuracy                           0.44      1000
   macro avg       0.44      0.44      0.44      1000
weighted avg       0.44      0.44      0.44      1000


Confusion Matrix:
[[ 86  38  24  21  31]
 [ 36 105  23   9  27]
 [ 40  23  66  36  35]
 [ 22  23  28 102  25]
 [ 35  26  27  29  83]]


In [ ]:
# ==============================
# Step 5: Predict face shape names
# ==============================
class_labels = {v: k for k, v in generator.class_indices.items()}

predicted_shapes = [class_labels[int(p)] for p in y_pred]
print("\n Example Predictions (first 10):")
print(predicted_shapes[:10])